In [1]:
import os
import sys
import random
import pickle
import json
from collections import Counter

import numpy as np
import pandas as pd

from rl_firm.env import BaselineABMEnv, make_sb_env, make_custom_env, EVALUATION_SEEDS

## Load learned firm eval stats from other notebooks

In [2]:
def load_pickle(path):
    with open(os.path.join(os.path.abspath(os.path.join('..')), path), "rb") as f:
        return pickle.load(f)

In [13]:
learned_stats, learned_models = load_pickle('stats/learned_eval_results.pkl')

In [4]:
STABLE_SEEDS = [20, 21, 22, 23, 24, 26, 27, 29, 31, 34, 35, 36, 37, 38, 39]
VIABLE_SEEDS = [20, 21, 22, 23, 24, 26, 27, 29, 30, 31, 34, 35, 36, 37, 38, 39]

## Simulation runs for stochastic baseline

In [5]:
def get_ranking(firm):
    return sorted(firm.model.firms,
                  key=lambda f: f.last_month_profit,
                  reverse=True).index(firm) + 1

def get_model_from_env(env):
    # Get the model from the wrapped model
    while True:
        if hasattr(env, '_model'):
            return env._model
        elif hasattr(env, 'env'):
            env = env.env
        else:
            raise Exception('Not ABM env')

def percentile_rank(a, x):
    a = np.asarray(a)
    if a.size == 0:
        raise ValueError("a must not be empty")
    below = np.sum(a < x)
    equal = np.sum(a == x)
    return 100.0 * (below + 0.5 * equal) / a.size

In [6]:
def stochastic_baseline_simulate(model_seed, baseline_seed, years=100):
    random.seed(baseline_seed)
    env = BaselineABMEnv()
    obs, info = env.reset(seed=model_seed)
    model = get_model_from_env(env)
    parameters = model.parameters
    learning_firm = model.learning_firms[0]
    stats = {
        # Firm stats
        'liquidity_buffers': [],
        'revenues': [],
        'profits': [],
        'rankings': [],
        'wage_rates': [],
        'goods_prices': [],
        'employee_counts': [],
        'customer_counts': [],
        'inventories': [],
        'last_month_demands': [],
        'months_since_hire_failure': [],
        # RL stats
        'wage_adjustments': [],
        'price_adjustments': [],
        'hr_actions': [],
        'rewards': [],
        # Analysis stats
        'marginal_costs': [],
        'price_markups': [],
        'inventory_to_demand': [],
        'price_percentiles': [],
        'wage_percentiles': [],
        'profit_percentiles': [],
        'employee_percentiles': [],
        'customer_percentiles': [],
        'median_active_prices': [],
        'median_active_wages': [],
        'median_active_profits': [],
        'median_active_employees': [],
        'median_active_customers': [],
        'active_firm_counts': [],
    }
    def collect_stats(action, reward):
        stats['liquidity_buffers'].append(learning_firm.liquidity_buffer)
        stats['revenues'].append(learning_firm.last_month_revenue)
        stats['profits'].append(learning_firm.last_month_profit)
        stats['rankings'].append(get_ranking(learning_firm))
        stats['wage_rates'].append(learning_firm.wage_rate)
        stats['goods_prices'].append(learning_firm.goods_price)
        stats['employee_counts'].append(learning_firm.employee_count)
        stats['customer_counts'].append(learning_firm.customer_count)
        stats['inventories'].append(learning_firm.inventories)
        stats['last_month_demands'].append(learning_firm.last_month_demand)
        stats['months_since_hire_failure'].append(learning_firm.months_since_hire_failure)
        # RL stats
        if 'hr_action' in action:
            stats['hr_actions'].append(int(np.clip(np.round(action['hr_action']), 0, 2)))
            stats['price_adjustments'].append(action['price_adjustment'])
            stats['wage_adjustments'].append(action['wage_adjustment'])
        else:
            stats['hr_actions'].append(int(np.clip(np.round(action[0]), 0, 2)))
            stats['price_adjustments'].append(action[1])
            stats['wage_adjustments'].append(action[2])
        stats['rewards'].append(reward)
        # Analysis stats
        stats['marginal_costs'].append(learning_firm.marginal_cost)
        stats['price_markups'].append(np.nan if learning_firm.marginal_cost <= 0 else learning_firm.goods_price / learning_firm.marginal_cost)
        stats['inventory_to_demand'].append(
            np.nan if learning_firm.last_month_demand <= 0 else
            learning_firm.inventories / learning_firm.last_month_demand
        )

        active_firms = model.firms.select(lambda f: not f.is_bankrupt)
        stats['price_percentiles'].append(percentile_rank(active_firms.get('goods_price'), learning_firm.goods_price))
        stats['wage_percentiles'].append(percentile_rank(active_firms.get('wage_rate'), learning_firm.wage_rate))
        stats['profit_percentiles'].append(percentile_rank(active_firms.get('last_month_profit'), learning_firm.last_month_profit))
        stats['employee_percentiles'].append(percentile_rank(active_firms.get('employee_count'), learning_firm.employee_count))
        stats['customer_percentiles'].append(percentile_rank(active_firms.get('customer_count'), learning_firm.customer_count))
        stats['median_active_prices'].append(np.median(active_firms.get('goods_price')))
        stats['median_active_wages'].append(np.median(active_firms.get('wage_rate')))
        stats['median_active_profits'].append(np.median(active_firms.get('last_month_profit')))
        stats['median_active_employees'].append(np.median(active_firms.get('employee_count')))
        stats['median_active_customers'].append(np.median(active_firms.get('customer_count')))
        stats['active_firm_counts'].append(len(active_firms))

    # Use random from random library for randomness
    def create_baseline_predictor():
        def predict_baseline(obs):
            wage_adjustment = 0
            # check if there was a open position last month (not fulfilled)
            if obs['open_position']:
                wage_adjustment = random.uniform(0, parameters.firm.delta_wage_adjustment_rate)
            
            # check if should decrease wage
            if obs['months_since_hire_failure'] >= parameters.firm.gamma_consecutive_months:
                wage_adjustment = - random.uniform(0, parameters.firm.delta_wage_adjustment_rate)
        
            price_adjustment = 0
            # Only set new price with probs theta_probs_set_new_price
            if random.random() < parameters.firm.theta_probs_set_new_price:
                # Consider increasing price if not enough inventories
                marginal_cost = obs['wage_rate'] / (parameters.firm.lambda_technology * parameters.month_length)
                if (obs['inventories'] < parameters.firm.lower_phi_demand * obs['last_month_demand'] and
                    obs['goods_price'] <= parameters.firm.upper_phi_marginal_cost * marginal_cost):
                    price_adjustment = random.uniform(0, parameters.firm.theta_goods_price)
            
                # Else consider reducing the price
                if (obs['inventories'] > parameters.firm.upper_phi_demand * obs['last_month_demand'] and
                    obs['goods_price'] >= parameters.firm.lower_phi_marginal_cost * marginal_cost):
                    price_adjustment = - random.uniform(0, parameters.firm.theta_goods_price)
            
            hr_action = 1
            # check if need to hire more
            if obs['inventories'] < parameters.firm.lower_phi_demand * obs['last_month_demand']:
                hr_action = 2
            # or check if need to put someone on notice
            elif obs['inventories'] > parameters.firm.upper_phi_demand * obs['last_month_demand'] and obs['employee_count'] > 0:
                hr_action = 0
        
            return {
                'hr_action': hr_action,
                'price_adjustment': price_adjustment,
                'wage_adjustment': wage_adjustment
            }
    
        return predict_baseline

    predict_baseline = create_baseline_predictor()
    for _ in range(years * 12):
        action = predict_baseline(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        collect_stats(action, reward)
        if terminated:
            print('Terminated! Went belly up')
            break
        if truncated:
            print('Truncated! Unprofitable')
            break
    
    print(f'Firm survived for {len(stats['profits']) // 12} years, making a total amount of {sum(stats['profits'])} of profit, with total reward {sum(stats['rewards'])}')

    return stats, model

In [7]:
BASELINE_SEEDS = list(range(100, 120))
baseline_stochastic_stats, baseline_stochastic_models = {}, {}
for model_seed in EVALUATION_SEEDS:
    baseline_stochastic_stats[model_seed] = {}
    baseline_stochastic_models[model_seed] = {}
    for baseline_seed in BASELINE_SEEDS:
        stat, model = stochastic_baseline_simulate(model_seed=model_seed,
                                                   baseline_seed=baseline_seed)
        baseline_stochastic_stats[model_seed][baseline_seed] = stat
        baseline_stochastic_models[model_seed][baseline_seed] = model

Terminated! Went belly up
Firm survived for 6 years, making a total amount of 496139.2744969281 of profit, with total reward 1.1363775320942415
Terminated! Went belly up
Firm survived for 14 years, making a total amount of 729157.2548133971 of profit, with total reward 1.6853101555606342
Terminated! Went belly up
Firm survived for 11 years, making a total amount of 687194.1201602757 of profit, with total reward 1.5780544417077504
Terminated! Went belly up
Firm survived for 26 years, making a total amount of 1501554.692617513 of profit, with total reward 3.497228368322534
Terminated! Went belly up
Firm survived for 31 years, making a total amount of 3203148.18546175 of profit, with total reward 7.068379789567689
Terminated! Went belly up
Firm survived for 81 years, making a total amount of 6841726.414942538 of profit, with total reward 15.15953543578527
Terminated! Went belly up
Firm survived for 50 years, making a total amount of 3413485.9808404665 of profit, with total reward 7.703441

In [8]:
# Save simulation results
with open(os.path.join(os.path.abspath(os.path.join('..')), f'stats/baseline_stochastic_eval_results.pkl'), "wb") as f:
    pickle.dump((baseline_stochastic_stats, baseline_stochastic_models), f)

## Load the saved data

In [4]:
baseline_stochastic_stats, baseline_stochastic_models = load_pickle('stats/baseline_stochastic_eval_results.pkl')

## Analysis

In [15]:
def survival_years(stats):
    return len(stats["profits"]) / 12.0

def survived_full_horizon(stats, full_horizon_years=100.0):
    """
    Returns 1.0 if the run survived to the end of the simulation horizon, else 0.0.
    """
    return float(survival_years(stats) >= full_horizon_years)

def cumulative_profit(stats):
    return float(np.sum(stats["profits"]))

def zero_buffer_share(stats, horizon_months=120, pad_with_zeros=True):
    arr = np.asarray(stats["liquidity_buffers"], dtype=float)

    if len(arr) >= horizon_months:
        return float(np.mean(arr[:horizon_months] <= 1e-8))

    if pad_with_zeros:
        padded = np.zeros(horizon_months, dtype=float)
        padded[:len(arr)] = arr
        return float(np.mean(padded <= 1e-8))

    if len(arr) == 0:
        return np.nan

    return float(np.mean(arr <= 1e-8))

def under_target_buffer_share(stats, chi_buffer=0.1, horizon_months=120, eps=1e-8):
    buffers = np.asarray(stats["liquidity_buffers"][:horizon_months], dtype=float)
    wages = np.asarray(stats["wage_rates"][:horizon_months], dtype=float)
    employees = np.asarray(stats["employee_counts"][:horizon_months], dtype=float)

    n = min(len(buffers), len(wages), len(employees), horizon_months)
    if n == 0:
        return np.nan

    buffers = buffers[:n]
    wages = wages[:n]
    employees = employees[:n]

    target = chi_buffer * wages * employees
    valid = target > eps

    if np.sum(valid) == 0:
        return np.nan

    return float(np.mean(buffers[valid] < target[valid]))

def mean_buffer_to_target_ratio(stats, chi_buffer=0.1, horizon_months=120, eps=1e-8):
    buffers = np.asarray(stats["liquidity_buffers"][:horizon_months], dtype=float)
    wages = np.asarray(stats["wage_rates"][:horizon_months], dtype=float)
    employees = np.asarray(stats["employee_counts"][:horizon_months], dtype=float)

    n = min(len(buffers), len(wages), len(employees), horizon_months)
    if n == 0:
        return np.nan

    target = chi_buffer * wages[:n] * employees[:n]
    valid = target > eps

    if np.sum(valid) == 0:
        return np.nan

    return float(np.mean(buffers[:n][valid] / target[valid]))

def fixed_horizon_mean(values, horizon_months=120, pad_with_zeros=True):
    """
    Mean over a fixed horizon.
    If the run ends early and pad_with_zeros=True, remaining months are treated as 0.
    """
    arr = np.asarray(values, dtype=float)

    if len(arr) >= horizon_months:
        return float(np.mean(arr[:horizon_months]))

    if pad_with_zeros:
        padded = np.zeros(horizon_months, dtype=float)
        padded[:len(arr)] = arr
        return float(np.mean(padded))

    # alive-only alternative
    if len(arr) == 0:
        return np.nan
    return float(np.mean(arr))

def summarize_learned_seed(stats, horizon_months=120):
    return {
        "survived_full_horizon": survived_full_horizon(stats),
        "survival_years": survival_years(stats),
        "cumulative_profit": cumulative_profit(stats),
        "zero_buffer_share_10y": zero_buffer_share(stats, horizon_months=horizon_months),
        "under_target_buffer_share_10y": under_target_buffer_share(stats, horizon_months=horizon_months),
        "avg_buffer_to_target_ratio_10y": mean_buffer_to_target_ratio(stats, horizon_months=horizon_months),
        "avg_monthly_profit_10y": fixed_horizon_mean(stats["profits"], horizon_months),
        "avg_employee_count_10y": fixed_horizon_mean(stats["employee_counts"], horizon_months),
        "avg_customer_count_10y": fixed_horizon_mean(stats["customer_counts"], horizon_months),
        "avg_liquidity_buffer_10y": fixed_horizon_mean(stats["liquidity_buffers"], horizon_months),
        "avg_goods_price_10y": fixed_horizon_mean(stats["goods_prices"], horizon_months, pad_with_zeros=False),
        "avg_wage_rate_10y": fixed_horizon_mean(stats["wage_rates"], horizon_months, pad_with_zeros=False),
    }

def summarize_baseline_seed(runs: dict, horizon_months=120) -> dict:
    """
    runs = {baseline_seed: stats_dict}
    Returns median baseline outcome within one model seed + IQR info.
    """
    surv = []
    survived_full = []
    cum = []
    zero_buf_10y = []
    ut_buf_10y = []
    buf_target_ratio_10y = []
    avg_profit_10y = []
    avg_emp_10y = []
    avg_cust_10y = []
    avg_buf_10y = []
    avg_price_10y = []
    avg_wage_10y = []

    for _, stats in runs.items():
        surv.append(survival_years(stats))
        survived_full.append(survived_full_horizon(stats))
        cum.append(cumulative_profit(stats))
        zero_buf_10y.append(zero_buffer_share(stats, horizon_months=horizon_months))
        ut_buf_10y.append(under_target_buffer_share(stats, horizon_months=horizon_months))
        buf_target_ratio_10y.append(mean_buffer_to_target_ratio(stats, horizon_months=horizon_months))
        avg_profit_10y.append(fixed_horizon_mean(stats["profits"], horizon_months))
        avg_emp_10y.append(fixed_horizon_mean(stats["employee_counts"], horizon_months))
        avg_cust_10y.append(fixed_horizon_mean(stats["customer_counts"], horizon_months))
        avg_buf_10y.append(fixed_horizon_mean(stats["liquidity_buffers"], horizon_months))
        avg_price_10y.append(fixed_horizon_mean(stats["goods_prices"], horizon_months, pad_with_zeros=False))
        avg_wage_10y.append(fixed_horizon_mean(stats["wage_rates"], horizon_months, pad_with_zeros=False))

    def med_q1_q3_iqr(x):
        x = np.asarray(x, dtype=float)
        q1 = float(np.percentile(x, 25))
        med = float(np.percentile(x, 50))
        q3 = float(np.percentile(x, 75))
        return med, q1, q3, q3 - q1

    surv_med, surv_q1, surv_q3, surv_iqr = med_q1_q3_iqr(surv)
    full_mean = float(np.mean(np.asarray(survived_full, dtype=float)))
    full_med, full_q1, full_q3, full_iqr = med_q1_q3_iqr(survived_full)
    cum_med, cum_q1, cum_q3, cum_iqr = med_q1_q3_iqr(cum)
    zero_buf_med, zero_buf_q1, zero_buf_q3, zero_buf_iqr = med_q1_q3_iqr(zero_buf_10y)
    ut_buf_med, ut_buf_q1, ut_buf_q3, ut_buf_iqr = med_q1_q3_iqr(ut_buf_10y)
    buft_ratio_med, buft_ratio_q1, buft_ratio_q3, buft_ratio_iqr = med_q1_q3_iqr(buf_target_ratio_10y)
    prof_med, prof_q1, prof_q3, prof_iqr = med_q1_q3_iqr(avg_profit_10y)
    emp_med, emp_q1, emp_q3, emp_iqr = med_q1_q3_iqr(avg_emp_10y)
    cust_med, cust_q1, cust_q3, cust_iqr = med_q1_q3_iqr(avg_cust_10y)
    buf_med, buf_q1, buf_q3, buf_iqr = med_q1_q3_iqr(avg_buf_10y)
    price_med, price_q1, price_q3, price_iqr = med_q1_q3_iqr(avg_price_10y)
    wage_med, wage_q1, wage_q3, wage_iqr = med_q1_q3_iqr(avg_wage_10y)

    return {
        "baseline_survived_full_horizon_mean": full_mean,
        "baseline_survived_full_horizon_median": full_med,
        "baseline_survived_full_horizon_q1": full_q1,
        "baseline_survived_full_horizon_q3": full_q3,
        "baseline_survived_full_horizon_iqr": full_iqr,
        
        "baseline_survival_median": surv_med,
        "baseline_survival_q1": surv_q1,
        "baseline_survival_q3": surv_q3,
        "baseline_survival_iqr": surv_iqr,

        "baseline_cumprofit_median": cum_med,
        "baseline_cumprofit_q1": cum_q1,
        "baseline_cumprofit_q3": cum_q3,
        "baseline_cumprofit_iqr": cum_iqr,

        "baseline_zero_buffer_share_10y_median": zero_buf_med,
        "baseline_zero_buffer_share_10y_q1": zero_buf_q1,
        "baseline_zero_buffer_share_10y_q3": zero_buf_q3,
        "baseline_zero_buffer_share_10y_iqr": zero_buf_iqr,

        "baseline_under_target_buffer_share_10y_median": ut_buf_med,
        "baseline_under_target_buffer_share_10y_q1": ut_buf_q1,
        "baseline_under_target_buffer_share_10y_q3": ut_buf_q3,
        "baseline_under_target_buffer_share_10y_iqr": ut_buf_iqr,

        "baseline_avg_buffer_to_target_ratio_10y_median": buft_ratio_med,
        "baseline_avg_buffer_to_target_ratio_10y_q1": buft_ratio_q1,
        "baseline_avg_buffer_to_target_ratio_10y_q3": buft_ratio_q3,
        "baseline_avg_buffer_to_target_ratio_10y_iqr": buft_ratio_iqr,

        "baseline_avg_monthly_profit_10y_median": prof_med,
        "baseline_avg_monthly_profit_10y_q1": prof_q1,
        "baseline_avg_monthly_profit_10y_q3": prof_q3,
        "baseline_avg_monthly_profit_10y_iqr": prof_iqr,

        "baseline_avg_employee_count_10y_median": emp_med,
        "baseline_avg_employee_count_10y_q1": emp_q1,
        "baseline_avg_employee_count_10y_q3": emp_q3,
        "baseline_avg_employee_count_10y_iqr": emp_iqr,

        "baseline_avg_customer_count_10y_median": cust_med,
        "baseline_avg_customer_count_10y_q1": cust_q1,
        "baseline_avg_customer_count_10y_q3": cust_q3,
        "baseline_avg_customer_count_10y_iqr": cust_iqr,

        "baseline_avg_liquidity_buffer_10y_median": buf_med,
        "baseline_avg_liquidity_buffer_10y_q1": buf_q1,
        "baseline_avg_liquidity_buffer_10y_q3": buf_q3,
        "baseline_avg_liquidity_buffer_10y_iqr": buf_iqr,

        "baseline_avg_goods_price_10y_median": price_med,
        "baseline_avg_goods_price_10y_q1": price_q1,
        "baseline_avg_goods_price_10y_q3": price_q3,
        "baseline_avg_goods_price_10y_iqr": price_iqr,

        "baseline_avg_wage_rate_10y_median": wage_med,
        "baseline_avg_wage_rate_10y_q1": wage_q1,
        "baseline_avg_wage_rate_10y_q3": wage_q3,
        "baseline_avg_wage_rate_10y_iqr": wage_iqr,
    }

In [17]:
def build_per_seed_df(learned_data, baseline_data, horizon_months=120):
    rows = []
    for model_seed in EVALUATION_SEEDS:
        if model_seed not in VIABLE_SEEDS:
            continue
    
        learned_summary = summarize_learned_seed(learned_data[model_seed-20], horizon_months)
        baseline_summary = summarize_baseline_seed(baseline_data[model_seed], horizon_months)

        row = {"model_seed": model_seed}
        row.update({f"learned_{k}": v for k, v in learned_summary.items()})
        row.update(baseline_summary)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("model_seed").reset_index(drop=True)

per_seed_df = build_per_seed_df(learned_stats, baseline_stochastic_stats)
per_seed_df

,model_seed,learned_survived_full_horizon,learned_survival_years,learned_cumulative_profit,learned_zero_buffer_share_10y,learned_under_target_buffer_share_10y,learned_avg_buffer_to_target_ratio_10y,learned_avg_monthly_profit_10y,learned_avg_employee_count_10y,learned_avg_customer_count_10y,...,baseline_avg_liquidity_buffer_10y_q3,baseline_avg_liquidity_buffer_10y_iqr,baseline_avg_goods_price_10y_median,baseline_avg_goods_price_10y_q1,baseline_avg_goods_price_10y_q3,baseline_avg_goods_price_10y_iqr,baseline_avg_wage_rate_10y_median,baseline_avg_wage_rate_10y_q1,baseline_avg_wage_rate_10y_q3,baseline_avg_wage_rate_10y_iqr
0,20,1.0,100.00,6.502009e+07,0.108333,0.175000,0.859608,10646.733053,39.658333,142.141667,...,2706.844373,841.365403,43.732644,43.529193,43.780446,0.251253,2211.639728,2199.987583,2231.428285,31.440702
1,21,1.0,100.00,5.517542e+07,0.158333,0.291667,0.789991,12125.454217,61.100000,207.066667,...,4268.328805,981.399462,43.501359,43.275646,43.832431,0.556785,2195.549743,2186.270692,2206.677568,20.406877
2,22,1.0,100.00,4.154287e+07,0.100000,0.150000,0.879120,6004.880147,23.733333,105.800000,...,5622.377575,1239.967199,43.825485,43.825485,43.825485,0.000000,2334.288362,2315.655765,2353.793948,38.138182
3,23,1.0,100.00,5.252588e+07,0.041667,0.200000,0.915809,16974.961845,75.500000,254.008333,...,1156.266281,516.007314,46.693120,46.397658,46.929945,0.532288,2354.663868,2336.946460,2372.543759,35.597299
4,24,1.0,100.00,5.402778e+07,0.033333,0.083333,0.946765,10588.911953,45.425000,195.666667,...,1198.845500,1052.170951,47.862874,47.403582,48.044419,0.640837,2435.737395,2393.790628,2463.705484,69.914856
5,26,1.0,100.00,5.767346e+07,0.058333,0.108333,0.913552,10560.231260,42.733333,242.783333,...,111.243336,64.658748,46.313437,45.983318,46.625272,0.641954,2445.009065,2385.074378,2472.092466,87.018088
6,27,1.0,100.00,5.134681e+07,0.150000,0.383333,0.706551,14131.227497,63.825000,200.608333,...,2440.138995,819.442608,44.827754,44.580791,45.207630,0.626840,2256.947492,2247.137551,2274.369320,27.231769
7,29,1.0,100.00,5.410950e+07,0.058333,0.100000,0.925830,7643.774062,33.266667,190.408333,...,5.458129,0.049429,43.723705,43.723705,43.723705,0.000000,2191.829486,2178.842927,2197.084847,18.241920
8,30,0.0,27.75,2.116919e+06,0.091667,0.166667,0.864816,7334.429697,27.008333,122.950000,...,4038.085107,908.966485,42.931731,42.930886,42.931731,0.000844,2163.243496,2150.662222,2175.614982,24.952760
9,31,1.0,100.00,6.445505e+07,0.050000,0.116667,0.922052,16652.574398,63.158333,251.800000,...,1256.045042,553.909959,47.758111,47.247299,47.956552,0.709254,2386.266340,2374.363586,2405.241178,30.877592


In [18]:
per_seed_df[["model_seed", "learned_avg_buffer_to_target_ratio_10y", "baseline_avg_buffer_to_target_ratio_10y_median"]]

,model_seed,learned_avg_buffer_to_target_ratio_10y,baseline_avg_buffer_to_target_ratio_10y_median
0,20,0.859608,0.991956
1,21,0.789991,0.979570
2,22,0.879120,1.000000
3,23,0.915809,0.950811
4,24,0.946765,0.912583
5,26,0.913552,0.884151
6,27,0.706551,0.989437
7,29,0.925830,1.000000
8,30,0.864816,1.000000
9,31,0.922052,0.958565


In [19]:
def build_aggregate_compare_table(per_seed_df):
    rows = [
        (
            "Full-horizon survival rate (%)",
            100 * per_seed_df["learned_survived_full_horizon"].mean(),
            100 * per_seed_df["baseline_survived_full_horizon_mean"].mean(),
        ),
        (
            "Survival length (years)",
            f'{per_seed_df["learned_survival_years"].mean():.3f}',
            f'{per_seed_df["baseline_survival_median"].mean():.3f}',
        ),
        (
            "Cumulative profit",
            per_seed_df["learned_cumulative_profit"].mean(),
            per_seed_df["baseline_cumprofit_median"].mean(),
        ),
        (
            "Avg monthly profit, first 10 years",
            per_seed_df["learned_avg_monthly_profit_10y"].mean(),
            per_seed_df["baseline_avg_monthly_profit_10y_median"].mean(),
        ),
        (
            "Avg employee count, first 10 years",
            per_seed_df["learned_avg_employee_count_10y"].mean(),
            per_seed_df["baseline_avg_employee_count_10y_median"].mean(),
        ),
        (
            "Avg customer count, first 10 years",
            per_seed_df["learned_avg_customer_count_10y"].mean(),
            per_seed_df["baseline_avg_customer_count_10y_median"].mean(),
        ),
        (
            "Avg liquidity buffer, first 10 years",
            per_seed_df["learned_avg_liquidity_buffer_10y"].mean(),
            per_seed_df["baseline_avg_liquidity_buffer_10y_median"].mean(),
        ),
        (
            "Avg goods price, first 10 years",
            per_seed_df["learned_avg_goods_price_10y"].mean(),
            per_seed_df["baseline_avg_goods_price_10y_median"].mean(),
        ),
        (
            "Avg wage rate, first 10 years",
            per_seed_df["learned_avg_wage_rate_10y"].mean(),
            per_seed_df["baseline_avg_wage_rate_10y_median"].mean(),
        ),
    ]

    return pd.DataFrame(rows, columns=["Outcome", "Learned policy", "Baseline heuristic"])

compare_table_df = build_aggregate_compare_table(per_seed_df)
compare_table_df

,Outcome,Learned policy,Baseline heuristic
0,Full-horizon survival rate (%),93.75,0.625
1,Survival length (years),95.484,14.560
2,Cumulative profit,53405015.08596,1093510.049856
3,"Avg monthly profit, first 10 years",11539.165801,4909.02825
4,"Avg employee count, first 10 years",50.110937,10.113542
5,"Avg customer count, first 10 years",205.228646,67.93099
6,"Avg liquidity buffer, first 10 years",11549.970846,2320.251597
7,"Avg goods price, first 10 years",44.798872,45.310616
8,"Avg wage rate, first 10 years",2572.774874,2332.500729


In [20]:
print(compare_table_df.to_latex(index=False, escape=False))

\begin{tabular}{lll}
\toprule
Outcome & Learned policy & Baseline heuristic \\
\midrule
Full-horizon survival rate (%) & 93.750000 & 0.625000 \\
Survival length (years) & 95.484 & 14.560 \\
Cumulative profit & 53405015.085960 & 1093510.049856 \\
Avg monthly profit, first 10 years & 11539.165801 & 4909.028250 \\
Avg employee count, first 10 years & 50.110937 & 10.113542 \\
Avg customer count, first 10 years & 205.228646 & 67.930990 \\
Avg liquidity buffer, first 10 years & 11549.970846 & 2320.251597 \\
Avg goods price, first 10 years & 44.798872 & 45.310616 \\
Avg wage rate, first 10 years & 2572.774874 & 2332.500729 \\
\bottomrule
\end{tabular}



In [ ]:
baseline_stochastic_stats[